# Extra numbers for the paper

Sections D and K of `revision_experiments.ipynb` on their own. No model calls and no GPU: the notebook reads the experiment folder on Google Drive and streams the corpus from Hugging Face.

**Run:** Runtime > Run all, and allow access to Drive.

**Time:** section D reads the full corpus (more than two million rows), which takes from a few minutes to half an hour; the rest takes seconds.

**Output:** `revision_report_extra.txt`, saved in `runs/elner/revision/` and downloaded at the end.

In [ ]:
!pip install -q ijson huggingface_hub datasets

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Use elagent.py from this repository (cloned once into the Colab session).
import os, sys
REPO_URL = "https://github.com/MahammediAnesAbdelouahab/arabic-el-agent.git"
REPO_DIR = "/content/arabic-el-agent"
if not os.path.exists(os.path.join(REPO_DIR, "elagent.py")):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}

In [ ]:
import os, sys, io, contextlib, importlib, json, shutil, datetime
PROJECT_DIR = "/content/drive/MyDrive/arabic_el_agent"
sys.path.insert(0, REPO_DIR)
import elagent
importlib.reload(elagent)
from elagent import *
import pandas as pd
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)
print("elagent version:", ELAGENT_VERSION)
assert ELAGENT_VERSION >= "1.1.0", "old module loaded: Runtime > Restart session, then run again"

## Setup
No key and no model are needed here. Put your e-mail address in `user_agent`.

In [ ]:
CFG = Config(
    project_dir=PROJECT_DIR,
    llm_backend="gemini",
    llm_model="gemini-3.8-flash",        # the model used for every run in the paper
    min_llm_interval=0.0,
    user_agent="ArabicELAgent/0.1 (academic research; contact: your_email@example.com)",   # <- your e-mail
)
wd = WikidataClient(CFG.user_agent, cache_path=os.path.join(PROJECT_DIR, "cache", "wikidata_cache.json"))
# No LLM here: this notebook makes no model call.

EXP_DIR = os.path.join(PROJECT_DIR, "runs", "elner")
POOLED_CSV = os.path.join(EXP_DIR, "review_pooled.csv")
AGREE_CSV = os.path.join(EXP_DIR, "review_agreement_B.csv")
OUT_DIR = os.path.join(EXP_DIR, "revision")
os.makedirs(OUT_DIR, exist_ok=True)

all_rows = load_rows(os.path.join(PROJECT_DIR, "data", "elner_sample.jsonl"))
dev_rows, test_rows = split_rows(all_rows, dev_frac=0.2, seed=13)
print("dev:", len(dev_rows), "| test:", len(test_rows))      # expected: 211 | 850

KNOWN = set(ABLATIONS) | {"llm_plus_nil_rule"}


def list_runs(complete_only=False):
    """{run name: path} for every run file of the test set (dev runs and the stale memory run are left out)."""
    out = {}
    for f in sorted(os.listdir(EXP_DIR)):
        if not f.endswith(".jsonl") or f.startswith("dev_"):
            continue
        name = f[:-6]
        if name.split("@")[0] not in KNOWN or name.startswith("full_with_memory"):
            continue
        if complete_only:
            n = sum(1 for r in load_rows(os.path.join(EXP_DIR, f)) if r.get("qid") != "ERROR")
            if n < len(test_rows):
                print(f"not complete, left out: {name} ({n}/{len(test_rows)})")
                continue
        out[name] = os.path.join(EXP_DIR, f)
    return out


def first_runs(names):
    runs = list_runs()
    return {n: runs[n] for n in names if n in runs}


ACC, UNCLEAR, SUMM = pooled_gold(POOLED_CSV)
print("reviewed mentions:", len(ACC), "| unclear:", len(UNCLEAR))
print("runs found:", ", ".join(list_runs()))
REPORT = []          # every result below is also appended here, then written to revision_report.txt


def report(title, obj):
    text = obj.to_string() if hasattr(obj, "to_string") else json.dumps(obj, ensure_ascii=False, indent=1, default=str)
    REPORT.append(f"=== {title} ===\n{text}\n")
    print(f"=== {title} ===")
    display(obj) if hasattr(obj, "to_string") else print(text)

## D. Does the sample represent the corpus? (point 7)
The sample was drawn from a local copy of the Zenodo data file that ended early. This section:
1. counts the records read from the local copy and compares them with the 2,075,148 rows of the Hugging Face release;
2. streams the full corpus from Hugging Face, without loading it into memory;
3. compares three profiles (full corpus, part that was read, our sample): share of NIL labels, share of Latin script, entity types and sentence length;
4. checks that every sampled sentence appears in the Hugging Face release with the same text.

Reading the full corpus takes a few minutes.

In [ ]:
ELNER_DIR = os.path.join(PROJECT_DIR, "data", "elner_dz")
local_files = sorted(os.path.join(d, f) for d, _, fs in os.walk(ELNER_DIR) for f in fs
                     if f.lower().endswith((".json", ".jsonl", ".ndjson", ".csv", ".tsv", ".conll", ".txt", ".gz")))
print("local files:", local_files)
schema = detect_schema(list(iter_records(local_files[0], limit=300)))
print("schema:", schema)
prof_local = corpus_profile((r for f in local_files for r in iter_records(f)), schema)
print("records read from your copy:", prof_local["sentences"])

In [ ]:
HF_REPO = "HadjerHaninebgt7878/ELNER-DZ"
try:
    from datasets import load_dataset
    hf = load_dataset(HF_REPO, split="train", streaming=True)
    first_hf = next(iter(hf))
except Exception as e:                       # fallback: download the raw files and read them like the Zenodo copy
    print("datasets could not stream the repository:", repr(e)[:200], "-> downloading the files instead")
    from huggingface_hub import list_repo_files, hf_hub_download
    files = [f for f in list_repo_files(HF_REPO, repo_type="dataset") if f.lower().endswith((".json", ".jsonl", ".gz"))]
    print("data files:", files)
    paths = [hf_hub_download(HF_REPO, f, repo_type="dataset") for f in files]

    class _Stream:                            # re-iterable, like a streaming dataset
        def __iter__(self):
            return (r for p in paths for r in iter_records(p))
    hf = _Stream()
    first_hf = next(iter(hf))
print("first HF record:", json.dumps(first_hf, ensure_ascii=False)[:400])
schema_hf = detect_schema([r for _, r in zip(range(300), iter(hf))])
prof_full = corpus_profile(iter(hf), schema_hf)
prof_sample = corpus_profile(all_rows)
coverage = prof_local["sentences"] / prof_full["sentences"]
report("Corpus profiles (full = Hugging Face; read = your Zenodo copy; sample = our 1,000 sentences)",
       pd.DataFrame({"full corpus": prof_full, "read part": prof_local, "our sample": prof_sample}))
report("Coverage of your copy", {"records read": prof_local["sentences"], "records in full corpus": prof_full["sentences"],
                                 "coverage": round(coverage, 4)})

In [ ]:
# Are our sampled sentences in the Hugging Face version, with the same text?
want = {str(r["doc_id"]): r["text"] for r in all_rows}
found = same = 0
for rec in hf:
    rid = str(rec.get(schema_hf.get("id") or "id"))
    if rid in want:
        found += 1
        same += (rec.get(schema_hf["text"]) == want[rid])
        if found == len(want):
            break
report("Our sample in the Hugging Face version", {"sampled sentences": len(want), "found by id": found,
                                                   "identical text": same})

## K. Extra numbers
No model calls; a few seconds. **Run section D first in the same session** (it streams the corpus from Hugging Face), then this cell. It writes `revision_report_extra.txt` to `runs/elner/revision/`.

- What the rejected labels are: by type, and how many are years (`2015`, `2022`, ...).
- The main results again **without DATE mentions and years**, since ELNER-DZ leaves dates unlinked.
- Every repair and break in the agent, rule and Qwen runs, and the kinds of breaks of the general critic.

In [ ]:
import re, io, contextlib
EXTRA = []


def rep(title, obj):
    text = obj.to_string() if hasattr(obj, "to_string") else json.dumps(obj, ensure_ascii=False, indent=1, default=str)
    EXTRA.append(f"=== {title} ===\n{text}\n")
    print(f"=== {title} ===\n{text}\n")


ACC, UNCLEAR, SUMM = pooled_gold(POOLED_CSV)
runs = list_runs(complete_only=True)
row_of = {str(r["id"]): r for r in test_rows}
gold = {i: r["gold"] for i, r in row_of.items()}
t = pd.DataFrame({"id": list(row_of)})
t["type"] = [str(row_of[i].get("type") or "-") for i in t["id"]]
t["year"] = [bool(re.fullmatch(r"\s*[0-9\u0660-\u0669]{4}\s*", row_of[i]["mention"])) for i in t["id"]]
t["gold_nil"] = [gold[i] == "NIL" for i in t["id"]]
t["reviewed"] = [i in ACC for i in t["id"]]
t["unclear"] = [i in UNCLEAR for i in t["id"]]
t["gold_wrong"] = [i in ACC and gold[i] not in ACC[i] for i in t["id"]]
by_type = t.groupby("type")[["year", "gold_nil", "reviewed", "unclear", "gold_wrong"]].sum()
by_type.insert(0, "mentions", t.groupby("type").size())
rep("Test mentions by type", by_type)
rep("Wrong labels by original label (gold_nil), type and year",
    t[t.gold_wrong].groupby(["gold_nil", "type", "year"]).size().rename("n").reset_index())

DATES = set(t.loc[t["type"].str.upper().isin(["DATE", "TIME"]) | t["year"], "id"])
print("DATE/year mentions:", len(DATES))
MAIN = ["baseline_search_top1", "llm_no_correction", "llm_plus_nil_rule", "full_no_memory", "full_no_memory@r2",
        "grounded_only", "critic_only", "full_plus_role"]
main = {k: runs[k] for k in MAIN if k in runs}
qwen = {k: v for k, v in runs.items() if "qwen" in k.lower()}
for title, ex in (("all mentions", set(UNCLEAR)), ("without DATE/year mentions", set(UNCLEAR) | DATES)):
    with contextlib.redirect_stdout(io.StringIO()):
        rep(f"Corrected labels, {title}", paired_table(main, ref="llm_no_correction", acceptable=ACC, exclude=ex))
        if qwen:
            rep(f"Qwen, corrected labels, {title}", paired_table(qwen, acceptable=ACC, exclude=ex))
rep("What each check did in full_no_memory, without DATE/year mentions",
    fix_attribution(load_rows(runs["full_no_memory"]), acceptable=ACC, exclude=set(UNCLEAR) | DATES))


def changes(name):
    out = []
    for r in load_rows(runs[name]):
        i = str(r["id"])
        if i in UNCLEAR or r["qid"] == "ERROR":
            continue
        acc = ACC.get(i, {gold[i]})
        a, b = r.get("initial_qid") in acc, r["qid"] in acc
        if a != b:
            out.append({"run": name, "change": "repaired" if b else "broke", "id": i, "mention": r["mention"],
                        "type": r.get("type"), "initial": r.get("initial_qid"), "final": r["qid"],
                        "accepted": "|".join(sorted(acc))})
    return out


names = [n for n in ["full_no_memory", "full_no_memory@r2", "grounded_only", "llm_plus_nil_rule"] if n in runs] + list(qwen)
rep("Every repair and break (corrected labels)", pd.DataFrame([c for n in names for c in changes(n)]))
cc = pd.DataFrame(changes("critic_only")) if "critic_only" in runs else pd.DataFrame()
if len(cc):
    cc["kind"] = ["NIL -> QID" if a == "NIL" else "QID -> NIL" if b == "NIL" else "QID -> other QID"
                  for a, b in zip(cc["initial"], cc["final"])]
    rep("critic_only: repairs and breaks by kind (corrected labels)", cc.groupby(["change", "kind"]).size())

from_d = [x for x in REPORT if x.startswith(("=== Corpus profiles", "=== Coverage", "=== Our sample"))]
if not from_d:
    print("⚠️  section D (corpus profile) was not run in this session: run it, then this cell again")
path = os.path.join(OUT_DIR, "revision_report_extra.txt")
open(path, "w", encoding="utf-8").write(f"elagent {ELAGENT_VERSION} | {datetime.datetime.now():%Y-%m-%d %H:%M}\n\n"
                                        + "\n".join(from_d + EXTRA))
print("written:", path)

In [ ]:
# Download the report.
from google.colab import files
files.download(os.path.join(OUT_DIR, "revision_report_extra.txt"))